# cAPTure: Gate-0 data audit

Run this notebook in Google Colab with a **CPU runtime**. It audits development scenarios before selecting features, graph endpoints, or a window duration. It does not train models or access final-test scenarios.

The workflow uses mounted Google Drive for source CSVs and durable results, and local Colab disk for conversion and SQL aggregation. Process one scenario at a time. Start with `SMOKE`; `FULL_DEV` requires a reviewed smoke run.

The audit Parquet preserves every source row and raw column alongside diagnostic metadata. It is **not** a frozen, model-ready canonical dataset. No features are selected, labels guessed, or duplicate packets removed.


## 1. Mount Drive and load the project

Before running, push the implementation to the configured Git branch, or place an updated repository copy in Drive and set `PROJECT_ROOT` to that path. This notebook imports the repository module; it is not self-contained.

Source CSV file IDs and published names are still unresolved in the manifest. Use verified development CSVs already stored in Drive and configure their exact paths below. Do not use merge-notebook IDs, reduced feature datasets, or held-out author-train CSVs.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_ROOT = Path("/content/capture_gate0_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)], check=True
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_data.py",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")], check=True
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("CPU audit environment is ready.")


## 2. Run the small synthetic checks

Run these checks before downloading or processing large files. They cover half-open windows, repeated packets, cross-chunk timestamp inversions, unknown labels, held-out exclusion, and smoke-review integrity. They use temporary synthetic data and do not inspect cAPTure contents.

These checks have not been executed in the development workspace; this Colab run is the first runtime validation.


In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_data.py", "-v"],
    env=test_environment, cwd=PROJECT_ROOT, check=True,
)


## 3. Configure the audit and verified sources

Start with `SMOKE` (`train_empty_conn` and `train_dollar_char`). Paths below must point to the authors' full, pre-merged development CSVs. Set `metadata_verified=True` only after checking the scenario and author split. This verification is a recorded source-binding assertion, not an automatic content-provenance proof.

For `FULL_DEV`, supply all five paths and a smoke-review file from section 8. A unique run directory prevents overwriting earlier results. Leave the chunk size at 25,000 initially; reduce it if wide CSV columns exhaust RAM. DuckDB is limited to two threads and 2 GB, but pandas and Arrow also need memory.


In [ ]:
from datetime import datetime, timezone
import json
import shutil
import pandas as pd
from IPython.display import display
from utils.capture_data import (
    AuditSchema, inspect_csv, load_manifest, run_gate0, selected_scenarios,
    sha256_file, validate_smoke_review, write_json,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
MANIFEST = load_manifest(MANIFEST_PATH)
MODE = "SMOKE"
SCENARIOS = selected_scenarios(MANIFEST, MODE)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + MODE.lower()
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID

CHUNK_SIZE = 25_000
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = 2
KEEP_AUDIT_PARQUET = True
SMOKE_REVIEW_PATH = None  # Set the reviewed smoke JSON path before FULL_DEV.

# Replace None with exact full CSV paths in mounted Drive.
SOURCE_PATHS = {
    "train_empty_conn": None,
    "train_dollar_char": None,
    "train_qos_mid": None,
    "train_slash_char": None,
    "train_sub_exf": None,
}
SOURCE_METADATA_VERIFIED = False
SOURCES = {}
for scenario in SCENARIOS:
    configured_path = SOURCE_PATHS[scenario]
    if configured_path is None:
        raise ValueError(f"Set the verified Drive CSV path for {scenario}.")
    source_path = Path(configured_path)
    SOURCES[scenario] = {
        "drive_path": str(source_path),
        "expected_filename": source_path.name,
        "source_file_id": MANIFEST["scenarios"][scenario]["source_file_id"],
        "metadata_verified": SOURCE_METADATA_VERIFIED,
    }

if not SOURCE_METADATA_VERIFIED:
    raise ValueError("Verify the source bindings and set SOURCE_METADATA_VERIFIED=True.")
if MODE == "FULL_DEV":
    if SMOKE_REVIEW_PATH is None:
        raise ValueError("Set SMOKE_REVIEW_PATH before inspecting FULL_DEV sources.")
    validate_smoke_review(Path(SMOKE_REVIEW_PATH), sha256_file(MANIFEST_PATH), MANIFEST)

print(f"Mode: {MODE}")
print(f"Scenarios: {SCENARIOS}")
print(f"Output directory: {DRIVE_RUN_DIR}")
print(f"Free local storage: {shutil.disk_usage(LOCAL_ROOT).free / 1024**3:.1f} GiB")


## 4. Inspect the CSV headers and a bounded prefix

Only the first 2,000 rows are read here. Missing attack labels in this preview do not mean that the full scenario contains no attacks. Use the schema and source documentation to define the mapping in section 5.

All raw columns are initially read as strings to preserve identifiers, leading zeros, empty fields, and mixed types. Numeric parse diagnostics are produced during the full audit. The preview is not used to fit preprocessing or select features.


In [ ]:
CSV_SEPARATOR = ","
CSV_ENCODING = "utf-8-sig"
INSPECTIONS = {}
for scenario in SCENARIOS:
    source = SOURCES[scenario]
    source["separator"] = CSV_SEPARATOR
    source["encoding"] = CSV_ENCODING
    inspection = inspect_csv(
        Path(source["drive_path"]), separator=CSV_SEPARATOR,
        encoding=CSV_ENCODING, sample_rows=2_000,
    )
    INSPECTIONS[scenario] = inspection
    print(f"\n{scenario}: {inspection['source_size_bytes'] / 1024**3:.2f} GiB")
    display(pd.DataFrame([
        {"column": column, "prefix_examples": values}
        for column, values in inspection["sample_values"].items()
    ]))


## 5. Declare the schema explicitly

Replace every `None` below using the verified schema. Timestamp units must be one of `s`, `ms`, `us`, `ns`, or `datetime`. Raw label keys are strings after whitespace trimming. Include every admissible normal and attack label; unmatched labels remain unresolved and block progression.

The attack-step and phase fields may refer to the same source column only if the dataset annotations justify that interpretation. Endpoint identifiers are retained for topology diagnostics, never approved as model features here. Missing endpoints and missing malicious-packet annotations are reported as blockers.

The diagnostic iteration key is `(scenario, attack_step, phase, sequence_id)`. Its scientific meaning must be reviewed. A reused sequence identifier may require a different reconstruction rule.


In [ ]:
COMMON_SCHEMA = {
    "timestamp": None,
    "timestamp_unit": None,
    "label": None,
    "label_mapping": {},  # Example shape only: {"normal_value": 0, "attack_value": 1}.
    "source_endpoint": None,
    "destination_endpoint": None,
    "attack_step": None,
    "phase": None,
    "sequence_id": None,
    "separator": CSV_SEPARATOR,
    "encoding": CSV_ENCODING,
}
# Add per-scenario overrides only when the verified source schemas differ.
SCHEMA_OVERRIDES = {}
SCHEMAS = {}
for scenario in SCENARIOS:
    settings = {**COMMON_SCHEMA, **SCHEMA_OVERRIDES.get(scenario, {})}
    unresolved = [key for key, value in settings.items() if value is None]
    if unresolved:
        raise ValueError(f"Resolve schema fields for {scenario}: {unresolved}")
    schema = AuditSchema(**settings)
    schema.validate(INSPECTIONS[scenario]["columns"])
    SCHEMAS[scenario] = schema
print("All selected scenario mappings are explicit.")


## 6. Run the complete scenario audit

This cell scans each selected CSV completely. It stages one raw file locally, writes compressed audit Parquet, and computes disk-backed statistics for 1, 5, 10, and 30 seconds, including half-window origin shifts. Epoch-aligned boundaries are diagnostic candidates, not a frozen window choice.

Repeated packets remain distinct rows. Duplicate-row and duplicate-column detection uses SHA-256 fingerprints. Numeric parse failures may indicate legitimate categorical fields; inspect them before declaring malformed data. Window distributions describe occupied windows; empty windows between the first and last packet are counted separately.

Report files are copied to Drive and checksum-verified. Only then is that scenario's isolated local workspace deleted. Original Drive CSVs are preserved. A failure retains local work and records its location on Drive; a data-integrity blocker stops before the next scenario. SQL spill space can exceed raw size, so monitor local storage.

An audit with no automatic blockers still requires review of endpoints, sequence semantics, timestamps, duplicates, feature leakage, and window feasibility.


In [ ]:
RESULTS = run_gate0(
    manifest_path=MANIFEST_PATH,
    mode=MODE,
    sources=SOURCES,
    schemas=SCHEMAS,
    local_root=LOCAL_ROOT,
    drive_run_dir=DRIVE_RUN_DIR,
    smoke_review_path=Path(SMOKE_REVIEW_PATH) if SMOKE_REVIEW_PATH else None,
    chunksize=CHUNK_SIZE,
    memory_limit=DUCKDB_MEMORY_LIMIT,
    threads=DUCKDB_THREADS,
    keep_audit_parquet=KEEP_AUDIT_PARQUET,
)
print(json.dumps(RESULTS, indent=2))


## 7. Review the saved reports

The run directory contains the manifest snapshot, resolved runtime configuration, Git provenance, run status, and per-scenario artifacts:

- `audit_report.json`: schema, packet counts, raw labels, column diagnostics, blockers, and window summaries.
- `packets.audit.parquet`: all rows and raw fields, plus canonical diagnostic metadata (optional durable retention).
- `attack_iterations.parquet`: counts and durations grouped by attack step, phase, and sequence.
- `windows_*s_offset_*s.parquet`: occupied-window packet, node, and directed-pair counts.
- `artifact_checksums.json`: hashes of the persisted files.

A zero-duration attack iteration is possible when all its packets have the same timestamp. Duplicate benign data across scenarios sharing the same benign source is not removed; the manifest's background-separated folds address that evaluation risk.


In [ ]:
REPORTS = {}
for scenario, result in RESULTS.items():
    report = json.loads(Path(result["report"]).read_text())
    REPORTS[scenario] = report
    print(f"\n{scenario}: {report['status']}")
    print(f"Blockers: {report['blockers']}")
    display(pd.DataFrame([report["counts"]]))
    display(pd.DataFrame(report["raw_labels"]))
    window_summary = pd.DataFrame(report["windows"])
    window_summary["mean_packets_per_second_occupied"] = (
        window_summary["mean_packets_occupied"] / window_summary["width_seconds"]
    )
    display(window_summary)
    display(pd.DataFrame(report["column_profiles"]).T)
    print("Duplicate column groups:", report["duplicate_column_groups_sha256"])
    iterations = pd.read_parquet(Path(result["report"]).parent / "attack_iterations.parquet")
    display(iterations.head(50))


## 8. Record the smoke review before FULL_DEV

Do this only after both smoke scenarios finish and their scientific checks have been reviewed. Record the rationale and any unresolved choices. This review permits the full development audit; it does not freeze features, pass a modeling gate, or authorize final-test access.

The review binds the exact manifest and report hashes. Changing the manifest invalidates it. Keep `APPROVE_SMOKE=False` until review is complete. To run `FULL_DEV`, return to section 3, change the mode, configure all five source paths, and set `SMOKE_REVIEW_PATH` to the printed path.


In [ ]:
APPROVE_SMOKE = False
REVIEW_NOTES = ""

if APPROVE_SMOKE:
    if MODE != "SMOKE":
        raise ValueError("Create smoke reviews only from SMOKE runs.")
    expected = selected_scenarios(MANIFEST, "SMOKE")
    if set(REPORTS) != set(expected) or any(report["blockers"] for report in REPORTS.values()):
        raise ValueError("Both smoke reports must exist and have no automatic blockers.")
    if not REVIEW_NOTES.strip():
        raise ValueError("Record the review rationale before approval.")
    review = {
        "approved": True,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "review_notes": REVIEW_NOTES,
        "reports": {
            scenario: {
                "path": RESULTS[scenario]["report"],
                "sha256": sha256_file(Path(RESULTS[scenario]["report"])),
            }
            for scenario in expected
        },
    }
    review_path = DRIVE_RUN_DIR / "smoke_review.json"
    if review_path.exists():
        raise FileExistsError("A smoke review already exists; preserve the original decision.")
    write_json(review_path, review)
    validate_smoke_review(review_path, sha256_file(MANIFEST_PATH), MANIFEST)
    print(f"Smoke review saved: {review_path}")
else:
    print("Smoke review remains pending. No approval was recorded.")
